### 策略名称: 多维微观结构复合因子 (Multi-Dimensional Microstructure Composite Factor)

**策略概述:**

本策略综合利用高频快照数据中的多个维度信息，构建一个鲁棒的 15 分钟频率复合因子。
通过融合以下三个正交子因子，捕捉市场微观结构中不同层面的 Alpha 信号：

1. **加权订单簿不平衡因子 (Weighted Order Book Imbalance)**: 利用 5 档盘口的指数衰减加权买卖量不平衡度，
   结合买卖价差进行调整，衡量供需力量对比。
2. **日内均值回归因子 (Intraday Mean Reversion)**: 计算 15 分钟窗口内收盘价相对均价的偏离度，
   基于短期均值回归假设构建反转信号。
3. **量价相关性因子 (Price-Volume Correlation)**: 计算窗口内价格收益率与成交量变化的皮尔逊相关系数，
   捕捉资金流向与价格变动的同步性。

三个子因子分别经过标准化和 tanh 缩放后，按权重加权合成最终因子。

---

**数学逻辑:**

**子因子 1: 订单簿不平衡 (OBI)**
$$W_{bid} = \sum_{i=1}^{5} BidVol_i \times e^{-0.3(i-1)}$$
$$Imbalance = \frac{W_{bid} - W_{ask}}{W_{bid} + W_{ask}}$$
$$OBI = \frac{Imbalance}{\sqrt{|Spread_{rel}|} + \epsilon}$$

**子因子 2: 均值回归 (MR)**
$$MR = -\tanh\left(\frac{P_{close} - P_{avg}}{P_{avg}} \times 10\right)$$

**子因子 3: 量价相关 (PVC)**
$$PVC = Corr(r_t, \Delta V_t)$$

**复合因子:**
$$Factor = w_1 \times \tanh(Z_{OBI}) + w_2 \times MR + w_3 \times PVC$$

默认权重: $w_1 = 0.45, w_2 = 0.35, w_3 = 0.20$


In [ ]:
def main(datasource, start_date, end_date):
    """
    Multi-Dimensional Microstructure Composite Factor

    融合订单簿不平衡、日内均值回归、量价相关性三个子因子，
    构建 15 分钟频率的复合 Alpha 因子。

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    # 子因子权重
    w_obi = 0.45
    w_mr  = 0.35
    w_pvc = 0.20

    sql = f"""
    SET preserve_insertion_order=false;
    SET threads=4;

    WITH cte_snapshot AS (
        SELECT
            date,
            instrument_id,
            volume,

            -- 中间价
            (ask_price1 + bid_price1) / 2.0 AS mid_price,

            -- 交易日
            strftime(date, '%Y-%m-%d') AS trading_day,

            -- 加权 5 档买方量 (指数衰减)
            (
                COALESCE(bid_volume1, 0) * 1.0 +
                COALESCE(bid_volume2, 0) * EXP(-0.3) +
                COALESCE(bid_volume3, 0) * EXP(-0.6) +
                COALESCE(bid_volume4, 0) * EXP(-0.9) +
                COALESCE(bid_volume5, 0) * EXP(-1.2)
            ) AS weight_bid,

            -- 加权 5 档卖方量 (指数衰减)
            (
                COALESCE(ask_volume1, 0) * 1.0 +
                COALESCE(ask_volume2, 0) * EXP(-0.3) +
                COALESCE(ask_volume3, 0) * EXP(-0.6) +
                COALESCE(ask_volume4, 0) * EXP(-0.9) +
                COALESCE(ask_volume5, 0) * EXP(-1.2)
            ) AS weight_ask,

            -- 加权不平衡度
            (weight_bid - weight_ask) / (weight_bid + weight_ask + 1e-8) AS weighted_imbalance,

            -- 相对价差
            (ask_price1 - bid_price1) / ((ask_price1 + bid_price1) / 2.0 + 1e-8) AS relative_spread,

            -- 原始订单簿压力
            (weighted_imbalance / (sqrt(abs(relative_spread)) + 1e-8)) AS raw_pressure,

            -- 15 分钟时间分段
            CASE
                WHEN strftime(date, '%H%M') >= '0930' AND strftime(date, '%H%M') < '0945' THEN 94500
                WHEN strftime(date, '%H%M') >= '0945' AND strftime(date, '%H%M') < '1000' THEN 100000
                WHEN strftime(date, '%H%M') >= '1000' AND strftime(date, '%H%M') < '1015' THEN 101500
                WHEN strftime(date, '%H%M') >= '1015' AND strftime(date, '%H%M') < '1030' THEN 103000
                WHEN strftime(date, '%H%M') >= '1030' AND strftime(date, '%H%M') < '1045' THEN 104500
                WHEN strftime(date, '%H%M') >= '1045' AND strftime(date, '%H%M') < '1100' THEN 110000
                WHEN strftime(date, '%H%M') >= '1100' AND strftime(date, '%H%M') < '1115' THEN 111500
                WHEN strftime(date, '%H%M') >= '1115' AND strftime(date, '%H%M') <= '1130' THEN 113000
                WHEN strftime(date, '%H%M') >= '1300' AND strftime(date, '%H%M') < '1315' THEN 131500
                WHEN strftime(date, '%H%M') >= '1315' AND strftime(date, '%H%M') < '1330' THEN 133000
                WHEN strftime(date, '%H%M') >= '1330' AND strftime(date, '%H%M') < '1345' THEN 134500
                WHEN strftime(date, '%H%M') >= '1345' AND strftime(date, '%H%M') < '1400' THEN 140000
                WHEN strftime(date, '%H%M') >= '1400' AND strftime(date, '%H%M') < '1415' THEN 141500
                WHEN strftime(date, '%H%M') >= '1415' AND strftime(date, '%H%M') < '1430' THEN 143000
                WHEN strftime(date, '%H%M') >= '1430' AND strftime(date, '%H%M') < '1445' THEN 144500
                WHEN strftime(date, '%H%M') >= '1445' AND strftime(date, '%H%M') < '1457' THEN 150000
                ELSE -1
            END AS time_segment

        FROM {datasource}
        WHERE time_segment != -1
    ),

    -- 计算逐笔差分 (用于量价相关性 + 波动率)
    cte_delta AS (
        SELECT
            *,
            LAG(mid_price) OVER (PARTITION BY instrument_id, trading_day, time_segment ORDER BY date) AS prev_mid_price,
            LAG(volume) OVER (PARTITION BY instrument_id, trading_day, time_segment ORDER BY date) AS prev_volume,
            (mid_price - prev_mid_price) / (prev_mid_price + 1e-8) AS ret,
            (volume - prev_volume) AS vol_delta
        FROM cte_snapshot
    ),

    -- 15 分钟窗口聚合: 同时计算三个子因子
    cte_window AS (
        SELECT
            trading_day,
            time_segment,
            instrument_id,

            -- === 子因子 1: 订单簿不平衡 (OBI) ===
            avg(raw_pressure) AS obi_mean,
            nanstd(raw_pressure) AS obi_std,
            (last(raw_pressure) - obi_mean) / (obi_std + 1e-8) AS obi_z,

            -- 波动率 (用于 OBI 调整)
            nanstd(log(mid_price / (prev_mid_price + 1e-8))) AS volatility,
            CASE
                WHEN COUNT(*) > 10 THEN quantile(abs(ret), 0.8)
                ELSE 0.01
            END AS vol_threshold,
            CASE
                WHEN volatility > vol_threshold THEN obi_z * 0.7
                ELSE obi_z
            END AS obi_adjusted,
            tanh(obi_adjusted) AS sub_factor_obi,

            -- === 子因子 2: 均值回归 (MR) ===
            argMax(mid_price, date) AS close_mid,
            avg(mid_price) AS avg_mid,
            -1.0 * tanh(((close_mid - avg_mid) / (avg_mid + 1e-8)) * 10.0) AS sub_factor_mr,

            -- === 子因子 3: 量价相关 (PVC) ===
            CASE
                WHEN COUNT(CASE WHEN prev_mid_price IS NOT NULL AND prev_volume IS NOT NULL THEN 1 END) < 3 THEN 0.0
                ELSE COALESCE(CORR(ret, vol_delta), 0.0)
            END AS sub_factor_pvc,

            -- === 复合因子 ===
            ({w_obi} * sub_factor_obi + {w_mr} * sub_factor_mr + {w_pvc} * sub_factor_pvc) AS factor

        FROM cte_delta
        GROUP BY instrument_id, trading_day, time_segment
    )

    SELECT
        CAST(CONCAT(
            f.trading_day, ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.factor
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={'date': [start_date, end_date]}).df()
    return df


if __name__ == '__main__':
    """
    开发调试模块：分块并行回测引擎
    """
    from bigmodule import M
    import pandas as pd
    import structlog
    import gc
    from concurrent.futures import ThreadPoolExecutor, as_completed

    logger = structlog.get_logger()
    datasource = 'cpt_dwc_2026_stock_hs300_snapshot'

    full_start_date = '2023-01-01'
    full_end_date = '2024-12-01'

    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq='MS')
    chunk_params = []
    for start_dt in date_ranges:
        current_start = start_dt.strftime('%Y-%m-%d 00:00:00')
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d 23:59:59')
        chunk_params.append((current_start, current_end))

    all_results = []
    max_workers = 4

    logger.info(f'Starting Composite Factor Backtest: {full_start_date} to {full_end_date}')
    logger.info(f'Parallel Workers: {max_workers}')

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_date = {
            executor.submit(main, datasource, start, end): (start, end)
            for start, end in chunk_params
        }
        for future in as_completed(future_to_date):
            start, end = future_to_date[future]
            try:
                df_chunk = future.result()
                if df_chunk is not None and not df_chunk.empty:
                    all_results.append(df_chunk)
                    logger.info(f'Chunk Done: {start[:7]} | Rows: {len(df_chunk)}')
                else:
                    logger.warning(f'Chunk Empty: {start[:7]}')
                del df_chunk
                gc.collect()
            except Exception as e:
                logger.error(f'Error in chunk {start}: {e}')

    if all_results:
        logger.info('Concatenating all chunks...')
        final_data = pd.concat(all_results, ignore_index=True)
        final_data.sort_values(by=['date', 'instrument'], inplace=True)

        logger.info(f'All Done! Final Shape: {final_data.shape}')
        logger.info(f'Sample:\n{final_data.head()}')

        logger.info('Starting Evaluation...')
        try:
            _ = M.eval_dwc._latest(data=final_data)
        except Exception as e:
            logger.warning(f'Evaluation failed: {e}')
            print('Data preview:', final_data.head())
    else:
        logger.error('No data generated.')
